# 06. NCI-60 세포주별 독성 패턴 분석

**목표**: NCI-60 데이터로 암종 선택적 독성 화합물 발굴

**핵심 질문**: 어떤 화합물이 특정 암종에만 선택적으로 독성인가?

**분석 흐름**:
1. NCI-60 데이터 다운로드 + 정제
2. 암종별 평균 pGI50 계산
3. 선택성 지수 계산
4. 선택적 독성 화합물 발굴 + 시각화
5. SMILES 수집 + 분자 특성 분석


## 0. 라이브러리 로드

In [ ]:
import requests
import zipfile
import io
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from tqdm.notebook import tqdm
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors, Draw
from IPython.display import display

print('라이브러리 로드 완료')

## 1. NCI-60 데이터 다운로드

In [ ]:
# CellMiner에서 NCI-60 RAW 데이터 다운로드
url = 'https://discover.nci.nih.gov/cellminer/download/rawdataset/DTP_NCI60_RAW.zip'
print('다운로드 중... (약 21MB)')

r = requests.get(url, timeout=60)
print(f'상태: {r.status_code} | 크기: {len(r.content):,} bytes')

z = zipfile.ZipFile(io.BytesIO(r.content))
print(f'zip 내부: {z.namelist()}')

## 2. 데이터 로드 + 정제

In [ ]:
# header=9 로 로드
df_raw = pd.read_excel(
    z.open('output/DTP_NCI60_RAW.xlsx'),
    sheet_name='all',
    header=9
)
print(f'원본 shape: {df_raw.shape}')

# 세포주 컬럼 분리
cell_line_cols = [c for c in df_raw.columns if ':' in str(c)]
print(f'세포주 컬럼 수: {len(cell_line_cols)}개')

# 세포주 컬럼 숫자 변환
df_raw[cell_line_cols] = df_raw[cell_line_cols].apply(pd.to_numeric, errors='coerce')

# NSC # 기준으로 평균 계산
df_mean = df_raw.groupby('NSC #')[cell_line_cols].mean().reset_index()

# Drug name, PubChem SID 병합
meta = df_raw[['NSC #', 'Drug name', 'PubChem SID']].drop_duplicates('NSC #')
df_mean = df_mean.merge(meta, on='NSC #', how='left')

# 전체 평균 pGI50
df_mean['mean_pGI50'] = df_mean[cell_line_cols].mean(axis=1)

print(f'고유 화합물: {len(df_mean)}개')
print(f'pGI50 통계:')
print(df_mean['mean_pGI50'].describe().round(3))

## 3. 암종별 평균 pGI50 + 선택성 지수

In [ ]:
# 암종별 컬럼 분류
cancer_types = {
    'BR': [c for c in cell_line_cols if c.startswith('BR:')],
    'LC': [c for c in cell_line_cols if c.startswith('LC:')],
    'CO': [c for c in cell_line_cols if c.startswith('CO:')],
    'ME': [c for c in cell_line_cols if c.startswith('ME:')],
    'OV': [c for c in cell_line_cols if c.startswith('OV:')],
    'RE': [c for c in cell_line_cols if c.startswith('RE:')],
    'PR': [c for c in cell_line_cols if c.startswith('PR:')],
    'LE': [c for c in cell_line_cols if c.startswith('LE:')],
}

print('암종별 세포주 수:')
for ct, cols in cancer_types.items():
    print(f'  {ct}: {len(cols)}개')

# 암종별 평균 pGI50
for ct, cols in cancer_types.items():
    df_mean[f'mean_{ct}'] = df_mean[cols].mean(axis=1)

cancer_mean_cols = [f'mean_{ct}' for ct in cancer_types.keys()]

# 선택성 지수
df_mean['max_cancer'] = df_mean[cancer_mean_cols].max(axis=1)
df_mean['min_cancer'] = df_mean[cancer_mean_cols].min(axis=1)
df_mean['selectivity'] = df_mean['max_cancer'] - df_mean['min_cancer']

print(f'\n선택성 지수 통계:')
print(df_mean['selectivity'].describe().round(3))

## 4. 선택적 독성 화합물 발굴

In [ ]:
# 조건: 독성 강함(pGI50>6) AND 선택성 높음(selectivity>2)
df_selective = df_mean[
    (df_mean['mean_pGI50'] > 6) &
    (df_mean['selectivity'] > 2)
].copy()

# 민감/저항 암종 찾기
df_selective['sensitive_cancer'] = df_selective[cancer_mean_cols].idxmax(axis=1).str.replace('mean_', '')
df_selective['resistant_cancer'] = df_selective[cancer_mean_cols].idxmin(axis=1).str.replace('mean_', '')

print(f'선택적 고독성 화합물: {len(df_selective)}개')
print(f'(전체 {len(df_mean)}개 중 {len(df_selective)/len(df_mean):.1%})')

print('\n[민감 암종 분포]')
print(df_selective['sensitive_cancer'].value_counts())

print('\n[저항 암종 분포]')
print(df_selective['resistant_cancer'].value_counts())

print('\n=== 선택성 Top 10 ===')
top10 = df_selective.nlargest(10, 'selectivity')
print(top10[['NSC #', 'Drug name', 'mean_pGI50', 'selectivity',
             'sensitive_cancer', 'resistant_cancer']].to_string(index=False))

## 5. 시각화

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 왼쪽: 민감 암종 분포
df_selective['sensitive_cancer'].value_counts().plot(
    kind='bar', ax=axes[0], color='salmon', edgecolor='white'
)
axes[0].set_title('선택적 독성 화합물의\n민감 암종 분포', fontsize=12)
axes[0].set_xlabel('암종')
axes[0].set_ylabel('화합물 수')
axes[0].tick_params(axis='x', rotation=45)

# 가운데: 독성 강도 vs 선택성 scatter
axes[1].scatter(
    df_mean['mean_pGI50'], df_mean['selectivity'],
    alpha=0.2, c='steelblue', edgecolors='none', s=5
)
axes[1].scatter(
    df_selective['mean_pGI50'], df_selective['selectivity'],
    alpha=0.7, c='red', edgecolors='none', s=20,
    label=f'선택적 고독성 (n={len(df_selective)})'
)
axes[1].axvline(6, color='gray', linestyle='--', linewidth=1)
axes[1].axhline(2, color='gray', linestyle='--', linewidth=1)
axes[1].set_xlabel('평균 pGI50 (독성 강도)')
axes[1].set_ylabel('선택성 지수')
axes[1].set_title('독성 강도 vs 선택성\n(빨강=선택적 고독성)')
axes[1].legend()

# 오른쪽: Top 10
labels = []
for _, row in top10.iterrows():
    name = row['Drug name']
    nsc = row['NSC #']
    label = name[:20] if name != '-' else f'NSC {nsc}'
    labels.append(label)

axes[2].barh(range(10), top10['selectivity'].values,
             color='mediumpurple', edgecolor='white')
axes[2].set_yticks(range(10))
axes[2].set_yticklabels(labels, fontsize=9)
axes[2].set_xlabel('선택성 지수')
axes[2].set_title('선택성 Top 10 화합물')
axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig('../data/processed/selectivity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장 완료!')

## 6. 암종별 pGI50 히트맵 (Top 30 선택적 화합물)

In [ ]:
top30 = df_selective.nlargest(30, 'selectivity')

# 히트맵용 데이터
heatmap_data = top30[cancer_mean_cols].copy()
heatmap_data.columns = [c.replace('mean_', '') for c in cancer_mean_cols]

# 라벨
row_labels = []
for _, row in top30.iterrows():
    name = row['Drug name']
    nsc = row['NSC #']
    label = name[:25] if name != '-' else f'NSC {nsc}'
    row_labels.append(label)

heatmap_data.index = row_labels

plt.figure(figsize=(10, 12))
sns.heatmap(
    heatmap_data,
    cmap='RdYlGn',
    center=5,
    annot=True, fmt='.1f',
    linewidths=0.5,
    cbar_kws={'label': 'pGI50 (높을수록 독성 강함)'}
)
plt.title('암종별 pGI50 히트맵\n(선택성 Top 30 화합물)', fontsize=13)
plt.xlabel('암종')
plt.tight_layout()
plt.savefig('../data/processed/selectivity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장 완료!')

## 7. SMILES 수집 + 분자 특성 분석

In [ ]:
def get_smiles_batch(sids, timeout=30):
    """PubChem SID 배치 → SMILES"""
    sid_str = ','.join([str(int(s)) for s in sids if pd.notna(s)])
    try:
        # SID → CID
        r = requests.get(
            f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/substance/sid/{sid_str}/cids/JSON',
            timeout=timeout
        )
        if r.status_code != 200:
            return {}
        sid_to_cid = {
            info['SID']: info['CID'][0]
            for info in r.json()['InformationList']['Information']
            if 'CID' in info
        }
        # CID → SMILES
        cid_str = ','.join([str(c) for c in sid_to_cid.values()])
        r2 = requests.get(
            f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid_str}/property/CanonicalSMILES/JSON',
            timeout=timeout
        )
        if r2.status_code != 200:
            return {}
        cid_to_smiles = {
            p['CID']: p.get('CanonicalSMILES') or p.get('ConnectivitySMILES')
            for p in r2.json()['PropertyTable']['Properties']
        }
        return {sid: cid_to_smiles.get(cid) for sid, cid in sid_to_cid.items()}
    except:
        return {}

# 선택적 화합물 SMILES 수집
sids = df_selective['PubChem SID'].dropna().tolist()
print(f'SMILES 수집 대상: {len(sids)}개')

all_smiles = {}
for i in tqdm(range(0, len(sids), 100)):
    batch = sids[i:i+100]
    result = get_smiles_batch(batch)
    all_smiles.update(result)
    time.sleep(0.3)

df_selective['smiles'] = df_selective['PubChem SID'].map(
    lambda x: all_smiles.get(int(x)) if pd.notna(x) else None
)
print(f'SMILES 수집 완료: {df_selective["smiles"].notna().sum()}개')

In [ ]:
# 분자 특성 계산
def get_props(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    return {
        'MolWt': Descriptors.MolWt(mol),
        'LogP':  Descriptors.MolLogP(mol),
        'HBD':   rdMolDescriptors.CalcNumHBD(mol),
        'HBA':   rdMolDescriptors.CalcNumHBA(mol),
        'TPSA':  Descriptors.TPSA(mol),
        'Rings': rdMolDescriptors.CalcNumRings(mol),
    }

df_sel_smiles = df_selective[df_selective['smiles'].notna()].copy()
props = pd.DataFrame([get_props(s) for s in df_sel_smiles['smiles']])
df_sel_smiles = pd.concat([df_sel_smiles.reset_index(drop=True), props], axis=1)

print('=== 선택적 독성 화합물 분자 특성 ===')
print(df_sel_smiles[['MolWt','LogP','HBD','HBA','TPSA','Rings']].describe().round(2))

# 민감 암종별 분자 특성 비교
print('\n=== 민감 암종별 평균 LogP ===')
print(df_sel_smiles.groupby('sensitive_cancer')['LogP'].mean().round(2).sort_values(ascending=False))

## 8. 인사이트 요약

In [ ]:
print('=' * 60)
print('NCI-60 세포주별 독성 패턴 분석 인사이트')
print('=' * 60)

most_sensitive = df_selective['sensitive_cancer'].value_counts().index[0]
most_resistant = df_selective['resistant_cancer'].value_counts().index[0]

print(f'''
[1] 선택적 고독성 화합물
  전체 {len(df_mean)}개 중 {len(df_selective)}개 ({len(df_selective)/len(df_mean):.1%})
  조건: pGI50 > 6 (GI50 < 1μM) AND 선택성 지수 > 2

[2] 가장 많이 표적되는 암종: {most_sensitive}
  → 선택적 화합물이 {most_sensitive} 세포주에 가장 강하게 작용

[3] 가장 저항성 높은 암종: {most_resistant}
  → {most_resistant} 세포주는 상대적으로 독성에 덜 민감

[4] ADC 설계 관점
  선택성 지수 높은 화합물 = ADC 페이로드 후보
  특정 암종 타깃 ADC 설계 시 이 화합물들 우선 검토

[5] 한계
  - GI50은 IC50과 다른 지표 (성장억제 vs 세포독성)
  - 선택성 기준(>2)은 임의적 — 문헌 기준 검토 필요
  - 정상세포 데이터 없음 → 진정한 selectivity 측정 불가
''')

# 저장
import os
os.makedirs('../data/processed', exist_ok=True)
df_selective.to_csv('../data/processed/nci60_selective.csv', index=False)
print('저장 완료: data/processed/nci60_selective.csv')